In [ ]:
# Notebook 03 - Feature Engineering (step-by-step)

# Cell A - Setup & Config
# - Import libs: pandas, numpy, ast, sklearn, scipy.sparse, pathlib, json, joblib
# - Define config: input/output paths, seed, TF-IDF params, topk_similarity

# Cell B - Load processed data
# - Load data/processed/recipes_clean.csv
# - Load data/processed/interactions_clean.csv
# - Parse list-like columns: ingredients_clean, tags_clean

# Cell C - Schema checks & hard guards
# - Validate dtypes: id, user_id, recipe_id, rating, minutes, calories
# - Remove orphan interactions (recipe_id not in recipes)
# - Enforce feature ranges (minutes > 0, calories in [10, 5000] or flag outlier)

# Cell D - Anti-leakage split
# - Split interactions: train/validation (time-based or leave-last-1 per user)
# - Use train_interactions to compute user/popularity statistics
# - Do not use validation interactions when fitting user-level stats

# Cell E - Text feature construction
# - Build ingredients_text, tags_text, description_clean
# - Build combined_text with deterministic weighting strategy

# Cell F - TF-IDF recipe vectors
# - Fit TfidfVectorizer on combined_text
# - Build recipe_tfidf_matrix + recipe_id/index mapping

# Cell G - Content similarity artifacts
# - Compute top-k cosine neighbors per recipe (avoid full NxN persistence)
# - Output columns: recipe_id, neighbor_recipe_id, similarity

# Cell H - Numerical & constraint features
# - Build rule-based fields: calories, minutes, n_ingredients_calc
# - Build buckets/flags: quick_meal, low_calorie, etc.

# Cell I - User features (from train only)
# - avg_rating_given, rating_count, activity_level
# - favorite_tags from high-rated history
# - default fallback for new users

# Cell J - Recipe popularity features
# - rating_count, rating_mean, popularity_score (Bayesian smoothing)
# - is_cold_item flag for cold-start logic

# Cell K - Feature validation checklist
# - Validate nulls, schema, key uniqueness
# - Coverage checks: users/features, recipes/vectors, recipes/top-k neighbors
# - Print final sanity summary

# Cell L - Save artifacts + manifest
# - Save into artifacts/features/
#   tfidf_vectorizer.joblib
#   recipe_tfidf_matrix.npz
#   recipe_index_map.parquet
#   recipe_similarity_topk.parquet
#   recipe_features.parquet
#   user_features.parquet
#   popularity_features.parquet
#   feature_manifest.json

# Output
# - Print artifact paths, shapes, coverage, and build timestamp

In [5]:
import pandas as pd
import numpy as np
import re
import ast
import json
import joblib
from pathlib import Path
from datetime import datetime
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
df_recipes = pd.read_csv("../data/processed/recipes_clean.csv")
df_interactions = pd.read_csv("../data/processed/interactions_clean.csv")

def parse_list_safe(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x.strip())
            return v if isinstance(v, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

for col in ["ingredients_clean", "tags_clean", "ingredients", "tags", "steps"]:
    if col in df_recipes.columns:
        df_recipes[col] = df_recipes[col].apply(parse_list_safe)

print(f"recipes:      {df_recipes.shape}")
print(f"interactions: {df_interactions.shape}")
print(f"Sample ingredients_clean: {df_recipes['ingredients_clean'].iloc[0][:3]}")
print(f"Sample tags_clean:        {df_recipes['tags_clean'].iloc[0][:3]}")


recipes:      (77300, 24)
interactions: (706514, 5)
Sample ingredients_clean: ['winter squash', 'mexican seasoning', 'mixed spice']
Sample tags_clean:        ['60-minutes-or-less', 'time-to-make', 'course']


In [3]:
# === Cell C - Schema checks & hard guards ===

print("--- Before guards ---")
print(f"recipes:      {df_recipes.shape}")
print(f"interactions: {df_interactions.shape}")

# 1) Ép kiểu dữ liệu core
df_recipes["id"] = pd.to_numeric(df_recipes["id"], errors="coerce").astype("Int64")
df_recipes["minutes"] = pd.to_numeric(df_recipes["minutes"], errors="coerce")
df_recipes["calories"] = pd.to_numeric(df_recipes["calories"], errors="coerce")

df_interactions["user_id"] = pd.to_numeric(df_interactions["user_id"], errors="coerce").astype("Int64")
df_interactions["recipe_id"] = pd.to_numeric(df_interactions["recipe_id"], errors="coerce").astype("Int64")
df_interactions["rating"] = pd.to_numeric(df_interactions["rating"], errors="coerce")
if "date" in df_interactions.columns:
    df_interactions["date"] = pd.to_datetime(df_interactions["date"], errors="coerce")

# 2) Loại orphan interactions (recipe_id không có trong recipes)
valid_ids = set(df_recipes["id"].dropna())
n_before = len(df_interactions)
df_interactions = df_interactions[df_interactions["recipe_id"].isin(valid_ids)].copy()
print(f"Orphan interactions removed: {n_before - len(df_interactions):,}")

# 3) Filter range hợp lý cho feature
df_recipes = df_recipes[df_recipes["minutes"].between(1, 1440)].copy()
df_recipes["is_calorie_outlier"] = ~df_recipes["calories"].between(10, 5000)
n_outlier = df_recipes["is_calorie_outlier"].sum()
print(f"Calorie outliers flagged (kept but flagged): {n_outlier:,}")

# 4) Sync lại sau filter
df_interactions = df_interactions[df_interactions["recipe_id"].isin(set(df_recipes["id"]))].copy()

print("\n--- After guards ---")
print(f"recipes:      {df_recipes.shape}")
print(f"interactions: {df_interactions.shape}")
print(f"unique users: {df_interactions['user_id'].nunique():,}")
print(f"unique items: {df_interactions['recipe_id'].nunique():,}")
print(f"orphan check: {len(set(df_interactions['recipe_id']) - set(df_recipes['id']))} orphans")

--- Before guards ---
recipes:      (77300, 24)
interactions: (706514, 5)
Orphan interactions removed: 7,138
Calorie outliers flagged (kept but flagged): 692

--- After guards ---
recipes:      (77300, 25)
interactions: (699376, 5)
unique users: 30,836
unique items: 77,300
orphan check: 0 orphans


In [4]:
# === Cell D - Anti-leakage split ===
# Split interactions theo thời gian: 80% cũ nhất -> train, 20% mới nhất -> validation
# Chỉ dùng train_interactions để tính user/popularity stats phía sau

interactions_sorted = df_interactions.sort_values("date").reset_index(drop=True)
split_idx = int(len(interactions_sorted) * 0.8)

train_interactions = interactions_sorted.iloc[:split_idx].copy()
val_interactions = interactions_sorted.iloc[split_idx:].copy()

train_user_ids = set(train_interactions["user_id"])
train_recipe_ids = set(train_interactions["recipe_id"])
val_user_ids = set(val_interactions["user_id"])
val_recipe_ids = set(val_interactions["recipe_id"])

cold_start_users = val_user_ids - train_user_ids
cold_start_items = val_recipe_ids - train_recipe_ids

print(f"train_interactions: {len(train_interactions):,} rows")
print(f"val_interactions:   {len(val_interactions):,} rows")
print(f"train date range:   {train_interactions['date'].min()} -> {train_interactions['date'].max()}")
print(f"val   date range:   {val_interactions['date'].min()} -> {val_interactions['date'].max()}")
print(f"\ntrain users: {len(train_user_ids):,}  |  train items: {len(train_recipe_ids):,}")
print(f"val   users: {len(val_user_ids):,}  |  val   items: {len(val_recipe_ids):,}")
print(f"cold-start users in val: {len(cold_start_users):,}")
print(f"cold-start items in val: {len(cold_start_items):,}")

train_interactions: 559,500 rows
val_interactions:   139,876 rows
train date range:   2000-02-25 00:00:00 -> 2010-08-14 00:00:00
val   date range:   2010-08-14 00:00:00 -> 2018-12-19 00:00:00

train users: 27,021  |  train items: 72,860
val   users: 14,375  |  val   items: 44,122
cold-start users in val: 3,815
cold-start items in val: 4,440


In [ ]:
# === Cell E ===
# plan Cell E : Xác định các nguồn text trong dataset --> NormalizeText --> Convert list --> Tạo text
def normalize_text(text):
    """
    Normalize general text fields (description)
    """
    if pd.isna(text):
        return ""
    
    text = text.lower()
    
    # remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def normalize_ingredient(token):
    """
    Normalize ingredient tokens
    """
    if pd.isna(token):
        return ""
    
    token = token.lower().strip()
    
    # replace spaces with underscore
    token = token.replace(" ", "_")
    
    return token


def list_to_text(lst):
    """
    Convert list column to normalized text
    """
    if not isinstance(lst, list):
        return ""
    
    tokens = [normalize_ingredient(x) for x in lst]
    
    return " ".join(tokens)


def tags_to_text(lst):
    """
    Convert tags list to normalized text
    """
    if not isinstance(lst, list):
        return ""
    
    tokens = [normalize_text(x) for x in lst]
    
    return " ".join(tokens)


# ---------- Build text features ----------

# ingredients text
df_recipes["ingredients_text"] = df_recipes["ingredients_clean"].apply(list_to_text)

# tags text
df_recipes["tags_text"] = df_recipes["tags_clean"].apply(tags_to_text)

# description text
if "description" in df_recipes.columns:
    df_recipes["description_clean"] = df_recipes["description"].apply(normalize_text)
else:
    df_recipes["description_clean"] = ""


# ---------- Build combined_text with weighting ----------

def build_combined_text(row):
    
    ingredients = row["ingredients_text"]
    tags = row["tags_text"]
    description = row["description_clean"]
    
    combined = " ".join([
        ingredients,        # ingredients weight 1
        ingredients,        # ingredients weight 2
        tags,               # tags weight 1
        tags,               # tags weight 2
        description         # description weight 1
    ])
    
    return combined


df_recipes["combined_text"] = df_recipes.apply(build_combined_text, axis=1)


# ---------- Basic checks ----------

print("Text feature construction completed\n")

print("Example rows:\n")
display(
    df_recipes[
        ["id","ingredients_text","tags_text","description_clean","combined_text"]
    ].head()
)

print("\nNull counts:")
print(df_recipes[["ingredients_text","tags_text","combined_text"]].isnull().sum())

print("\nAverage text length:")
print(df_recipes["combined_text"].str.len().mean())

Text feature construction completed

Example rows:



,id,ingredients_text,tags_text,description_clean,combined_text
0,137739,winter_squash mexican_seasoning mixed_spice ho...,60 minutes or less time to make course main in...,autumn is my favorite time of year to cook thi...,winter_squash mexican_seasoning mixed_spice ho...
1,75452,sugar unsalted_butter bananas eggs fresh_lemon...,weeknight time to make course main ingredient ...,from ann hodgman s,sugar unsalted_butter bananas eggs fresh_lemon...
2,63986,lean_pork_chops flour salt dry_mustard garlic_...,weeknight time to make course main ingredient ...,here s and old standby i enjoy from time to ti...,lean_pork_chops flour salt dry_mustard garlic_...
3,43026,egg_roll_wrap whole_green_chilies cheese corns...,60 minutes or less time to make course main in...,a favorite from a local restaurant no longer i...,egg_roll_wrap whole_green_chilies cheese corns...
4,23933,butterscotch_chips chinese_noodles salted_peanuts,15 minutes or less time to make course prepara...,a little different and oh so good i include th...,butterscotch_chips chinese_noodles salted_pean...



Null counts:
ingredients_text    0
tags_text           0
combined_text       0
dtype: int64

Average text length:
844.5295601552393


In [8]:
print(df_recipes["combined_text"].isnull().sum())
print(df_recipes["combined_text"].sample(5))
print(df_recipes["combined_text"].str.split().explode().value_counts().head(10))

0
31606    potatoes fresh_thyme garlic_cloves half-and-ha...
53119    fresh_asparagus water sea_salt lemon extra_vir...
73547    ground_turkey plain_breadcrumbs egg green_onio...
32842    green_beans butter green_beans butter 15 minut...
29266    milk brown_sugar vanilla_extract hot_coffee gr...
Name: combined_text, dtype: str
combined_text
low            248594
or             239368
to             230029
less           201544
make           172018
main           169126
time           165680
preparation    155027
course         147578
and            121087
Name: count, dtype: int64


In [ ]:
# === Cell F ===
# Chuyển mỗi recipe thành vector TF-IDF 
# --> Tạo TF-IDF Vocabulary (tạo dictionary token → column index) --> Tạo TF-IDF Matrix (dùng scipy sparse matrix)
# --> Tạo mapping recipe_id ↔ matrix index (mapping recipe_id → matrix_index)

# Tham số cần có: max_features (Giới hạn vocabulary size), ngram_range, min_df, max_df

#output: TF-IDF model (tfidf_vectorizer), Recipe TF-IDF matrix(recipe_tfidf_matrix -> shape:(n_recipes, n_features)), Recipe index mapping (recipe_index_map)

#Artifacts cần lưu : tfidf_vectorizer.joblib  ;  recipe_tfidf_matrix.npz  ;  recipe_index_map.parquet